# Ejercicio 2: comparar con evidencia y elegir

El mismo `viajes_features.csv` y la misma preparación importada. Mismo esqueleto del Ejercicio 1, con los cinco modelos de la Semana 10 en un experimento propio. Al final eliges el mejor desde el registro, no desde tu memoria, y compruebas que el modelo guardado reproduce su número.

Los pasos 1 a 3 tienen `TODO`; los pasos 4 y 5 solo dicen qué variable dejar. La última celda imprime una comprobación por paso; no la modifiques. Corre todas las celdas una vez antes de empezar (**Run All**). Si vuelves a correr las celdas que entrenan, aparecen corridas repetidas en el registro: es lo esperado.

In [ ]:
import hashlib
import logging
import time

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from preparacion_taxis import dividir, preparacion

# MLflow avisa en cada corrida que el modelo se guarda sin firma (la
# descripcion de sus columnas de entrada). La firma importa al servir el
# modelo, en la Semana 12; hoy solo ensucia la salida.
logging.getLogger("mlflow").setLevel(logging.ERROR)

viajes = pd.read_csv("viajes_features.csv")
X_ent, X_prueba, y_ent, y_prueba = dividir(viajes)
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)


def huella(ruta):
    with open(ruta, "rb") as archivo:
        return hashlib.md5(archivo.read()).hexdigest()[:12]


huella_datos = huella("viajes_features.csv")
EXPERIMENTO = "tarifa-taxis-comparacion"

## Paso 1: el experimento

In [ ]:
# El registro vive en el servidor de MLflow del Codespace (puerto 5050). La
# direccion viene de la variable de entorno MLFLOW_TRACKING_URI, como en un
# equipo de datos: el codigo no la escribe. Sin esa variable (en tu laptop),
# MLflow guarda el registro en una carpeta mlruns al lado del notebook.
print("Registro de MLflow:", mlflow.get_tracking_uri())

# TODO 1: usa el experimento EXPERIMENTO.

## Paso 2: entrenar y registrar

In [ ]:
def evaluar(nombre, modelo):
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])

    # TODO 2: abre la corrida con el nombre del modelo y registra sus
    # hiperparametros, la semilla de la division (42), la huella de los
    # datos y las filas de entrenamiento.

    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
    mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))

    # TODO 3: registra las metricas mae_entrenamiento, mae_prueba y
    # segundos, y el pipeline entrenado como el artefacto "modelo".

    mlflow.end_run()
    print(f"{nombre:20s} entrenamiento {mae_ent:7.4f}   prueba {mae_prueba:7.4f}")

## Paso 3: los cinco modelos de la Semana 10

In [ ]:
# TODO 4: registra los cinco modelos de la Semana 10, con estos nombres:
# "linea base" (DummyRegressor con la mediana), "ridge" (Ridge con
# alpha=10.0), "bosque aleatorio" (200 arboles, random_state=42),
# "gradient boosting" (random_state=42) y "red neuronal" (MLPRegressor con
# una capa de 64 neuronas, max_iter=1000, random_state=42).

## Paso 4: el registro, ordenado

In [ ]:
# Deja en corridas las corridas del experimento (mlflow.search_runs),
# ordenadas de menor a mayor mae_prueba, y en mejor_run_id el run_id de la
# primera.
corridas = None
mejor_run_id = None

## Paso 5: marcar el ganador y reproducirlo

In [ ]:
# Marca la corrida ganadora con la etiqueta "candidato" = "final"
# (mlflow.MlflowClient().set_tag). Despues carga su modelo desde el
# registro con mlflow.sklearn.load_model("runs:/" + mejor_run_id + "/modelo"),
# calcula su error absoluto medio sobre X_prueba y dejalo en
# mae_reproducido.
mae_reproducido = None

## Comprobación

In [ ]:
def comprobar(numero, condicion):
    if condicion:
        print(f"Paso {numero}: OK")
    else:
        print(f"Paso {numero}: revisa")


experimento = mlflow.get_experiment_by_name(EXPERIMENTO)
registradas = None
if experimento is not None:
    registradas = mlflow.search_runs(experiment_ids=[experimento.experiment_id])
nombres = set()
if registradas is not None and len(registradas) > 0:
    nombres = set(registradas["tags.mlflow.runName"])

comprobar(1, experimento is not None)
comprobar(2, registradas is not None and "params.huella_datos" in registradas and (registradas["params.huella_datos"] == huella_datos).all())
comprobar(3, {"linea base", "ridge", "bosque aleatorio", "gradient boosting", "red neuronal"} <= nombres and registradas["metrics.mae_prueba"].notna().all())
comprobar(4, corridas is not None and mejor_run_id == corridas.loc[0, "run_id"] and corridas.loc[0, "tags.mlflow.runName"] == "red neuronal")
etiqueta = None
if mejor_run_id is not None:
    etiqueta = mlflow.MlflowClient().get_run(mejor_run_id).data.tags.get("candidato")
comprobar(5, etiqueta == "final" and mae_reproducido is not None and abs(mae_reproducido - corridas.loc[0, "metrics.mae_prueba"]) < 1e-9)

## Si terminas antes

Abre la interfaz de MLflow (pestaña **Ports** de VS Code, puerto 5050, ícono del globo), entra al experimento `tarifa-taxis-comparacion`, selecciona las cinco corridas y compáralas con **Compare**. Además del error, ¿qué te dice el registro de la red neuronal que no te diría una tabla escrita a mano?